# DrugResponse — pCR prediction from scratch

**Goal:** predict pathological complete response (**pCR**) to neoadjuvant chemotherapy in breast cancer from clinical features + tumor gene expression.

**Plan**
1. Load data (GEO series matrix) → `X`, `y`
2. Baseline model: clinical features only
3. Clinical + genes, with **in-fold feature selection** (no leakage)
4. Nested cross-validation → ROC-AUC with 95% CI
5. Interpretation (SHAP) and limitations

## 0. Setup
Run the cells below once per session (`Runtime → Run all` or Shift+Enter).

In [ ]:
import sys, gzip, io
import numpy as np
import pandas as pd
import sklearn, xgboost

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)

print("python ", sys.version.split()[0])
print("pandas ", pd.__version__)
print("sklearn", sklearn.__version__)
print("xgboost", xgboost.__version__)

### Download the data from GEO
Colab has internet access, so we fetch the two cohorts straight from NCBI GEO. Files live in `data/raw/` and disappear when the runtime resets — just re-run this cell.

In [ ]:
!mkdir -p data/raw
!wget -q -nc -P data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/GSE25065/matrix/GSE25065_series_matrix.txt.gz
!wget -q -nc -P data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE20nnn/GSE20194/matrix/GSE20194_series_matrix.txt.gz
!ls -lh data/raw

### Peek inside the file
Metadata lines start with `!`. Patient characteristics are in `!Sample_characteristics_ch1` lines; every cell is a `"key: value"` pair for one patient (usually, but not always, the same key across a line).

In [ ]:
PATH = "data/raw/GSE25065_series_matrix.txt.gz"

with gzip.open(PATH, "rt") as f:
    for i, line in enumerate(f):
        if line.startswith("!Sample_characteristics_ch1"):
            print(i, line[:120])
        if line.startswith("!series_matrix_table_begin"):
            print(i, "<-- expression table starts here")
            break

---
## Task 1. Load clinical data and target

**Logic (read before coding)**

1. Go through the file line by line. For every line starting with `!Sample_characteristics_ch1`:
   - split by tab `\t`, drop the first element (the line name), strip quotes `"`;
   - each element looks like `"er_status_ihc: P"` → split by `": "` into **key** and **value**;
   - the *i*-th element belongs to the *i*-th patient → store `value` under `key` **for that patient**.
   - ⚠️ Do **not** assume all elements in one line share the same key. It happens to hold for GSE25065, but in GSE20194 patients with a missing field have their later fields shifted to another line — taking the key from the first cell silently scrambles their data.
2. Patient IDs are in the line `!Sample_geo_accession` → use them as the **index**.
3. Result: DataFrame, rows = patients, columns = characteristics (a dict of dicts → `pd.DataFrame.from_dict(..., orient="index")`).
4. **Target:** `y = 1` if `pathologic_response_pcr_rd == "pCR"`, else `0`.

⚠️ **Leakage trap:** `dlda30_prediction`, `rcb_0_i_prediction`, `chemosensitivity_prediction` are *other models' predictions of pCR*, and `pathologic_response_rcb_class` is measured *after surgery*. They must **never** be used as features.

In [ ]:
def parse_characteristics(path):
    """Return a DataFrame: rows = patients (GSM IDs), columns = clinical characteristics."""
    # TODO: your code here
    pass


clinical = parse_characteristics(PATH)
clinical.head()

In [ ]:
# TODO: build the target y (0/1) from 'pathologic_response_pcr_rd'
y = None

### Check yourself

In [ ]:
assert clinical is not None and len(clinical) > 100, "clinical looks empty"
assert set(y.unique()) <= {0, 1}, "y must be 0/1"
assert (clinical.index == y.index).all(), "index of clinical and y must match"
print("patients:", len(y), "| pCR rate:", round(y.mean(), 3))   # expect ~0.2